In [ ]:
# Cell 1: Clone code (Kaggle: bật Internet trong Settings). Nhánh phải đã push code mới nhất
!git clone --single-branch --branch HoangMinh https://github.com/VanHung-05/zalo_challenge_2026_datamate.git
%cd zalo_challenge_2026_datamate
!git log --oneline -1

In [ ]:
# Cell 2: Cài thư viện, trừ llama-cpp-python (cài bản CUDA ở cell sau, tránh build bản CPU vô ích)
# Dùng sys.executable để pip cài vào ĐÚNG Python mà các cell sau dùng để chạy runner
import sys
!grep -v '^llama-cpp-python' requirements.txt > /tmp/requirements_no_llama.txt
!{sys.executable} -m pip install -q -r /tmp/requirements_no_llama.txt
!{sys.executable} -m pip install -q diskcache jinja2 typing-extensions  # dependencies của llama-cpp-python (cell sau dùng --no-deps)

In [ ]:
!pip install llama-cpp-python --no-deps --force-reinstall --no-cache-dir \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!python -c "import llama_cpp; print(llama_cpp.__version__, llama_cpp.llama_supports_gpu_offload())"

## Download model

In [ ]:
!{sys.executable} scripts/download_model.py

## Test inference

In [ ]:
# Chạy thử 5 câu train; đầu output phải có "[LLM] llama.cpp device: GPU". Xem thời gian/câu (elapsed_seconds) trước khi chạy cả bộ
!{sys.executable} src/datamate/runner.py -c configs/dev.yaml -d train --limit 5 -o outputs/train_5/submission.jsonl
!{sys.executable} scripts/evaluate.py -p outputs/train_5/submission.jsonl -g train/answers.jsonl --only-predicted

In [ ]:
# Xem log 1 câu: SQL model sinh ra, lỗi, kết quả
import glob, json
log = json.load(open(sorted(glob.glob('outputs/train_5/logs/*.json'))[0], encoding='utf-8'))
print(log['question'], '->', log['final'], f"({log['elapsed_seconds']}s)")
for a in log.get('attempts', []):
    print('--- attempt', a['attempt'], '| error:', a.get('error'))
    print(a.get('sql'))

## Inference public test

In [ ]:
# Chạy toàn bộ public test (1000 câu), ghi từng dòng nên dừng giữa chừng vẫn giữ kết quả đã chạy
!{sys.executable} src/datamate/runner.py -c configs/dev.yaml -d public_test -o outputs/public/submission.jsonl
!{sys.executable} scripts/validate_submission.py -p outputs/public/submission.jsonl -q public_test/questions.jsonl

In [ ]:
# Copy file nộp ra /kaggle/working để tải về từ tab Output
!cp outputs/public/submission.jsonl /kaggle/working/submission.jsonl && ls -la /kaggle/working/submission.jsonl